---
title: "Practice Activity 13-2: Ensemble Methods"
author: "Shiqi Wu"
format:
  html:
    embed-resources: true
    code-overflow: wrap
---

# Ensemble Methods


In [1]:
import pandas as pd

import numpy as np

## Choose your own Regression

**Working together with your partner**, choose an interesting data set to fit *regression* models for, that is, a data set with a *quantitative* target variable and several feature variables. Could be a data set that: we have used in class, we have used on an assignment, a data set from your project, you used in another course, or you just find interesting! You'll work together with your partner to develop *regression* models to predict the *quantitative* target (so you need to decide on the data set and target variable together with your partner).

After you have decided on the data set and target variable:

- One partner should fit a linear regression model
- One partner should fit a k-nearest neighbors regression model

If you're using an example from earlier in the class, you can use a model you have already come up with or you can try something new. Note: you do NOT have to agree on the features included in your models.

After each partner has their model, **work together to combine your code and create two more models**:

- An ensemble model that uses **voting**
- An ensemble model that uses **stacking**

For each of the four models (linear regression, knn regression, voting ensemble, stacking ensemble)

1. Fit the model using the training data
1. Use the fitted model to predict the target for a few observations of your choice, and compare the predictions
1. Estimate the test RMSE using cross-validation, and compare the results

Which model seems to work best?


In [2]:
df_housing = pd.read_csv(
    "https://dlsun.github.io/pods/data/AmesHousing.txt",
    sep="\t"
)

reg_features = ["Gr Liv Area", "Bedroom AbvGr", "Full Bath"]

X_reg = df_housing[reg_features]
y_reg = df_housing["SalePrice"]

df_housing[reg_features + ["SalePrice"]].head()

,Gr Liv Area,Bedroom AbvGr,Full Bath,SalePrice
0,1656,3,1,215000
1,896,2,1,105000
2,1329,3,1,172000
3,2110,3,2,244000
4,1629,3,2,189900


In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

linear_reg_model = LinearRegression()

knn_reg_model = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor(n_neighbors=10)
)

linear_reg_model.fit(X_reg, y_reg)
knn_reg_model.fit(X_reg, y_reg)

linear_reg_model, knn_reg_model

(LinearRegression(),
 Pipeline(steps=[('standardscaler', StandardScaler()),
                 ('kneighborsregressor', KNeighborsRegressor(n_neighbors=10))]))

In [4]:
from sklearn.ensemble import VotingRegressor, StackingRegressor

voting_reg_model = VotingRegressor(
    estimators=[
        ("linear", linear_reg_model),
        ("knn", knn_reg_model)
    ]
)

stacking_reg_model = StackingRegressor(
    estimators=[
        ("linear", linear_reg_model),
        ("knn", knn_reg_model)
    ],
    final_estimator=LinearRegression(),
    cv=5
)

voting_reg_model.fit(X_reg, y_reg)
stacking_reg_model.fit(X_reg, y_reg)

voting_reg_model, stacking_reg_model

(VotingRegressor(estimators=[('linear', LinearRegression()),
                             ('knn',
                              Pipeline(steps=[('standardscaler',
                                               StandardScaler()),
                                              ('kneighborsregressor',
                                               KNeighborsRegressor(n_neighbors=10))]))]),
 StackingRegressor(cv=5,
                   estimators=[('linear', LinearRegression()),
                               ('knn',
                                Pipeline(steps=[('standardscaler',
                                                 StandardScaler()),
                                                ('kneighborsregressor',
                                                 KNeighborsRegressor(n_neighbors=10))]))],
                   final_estimator=LinearRegression()))

In [5]:
example_houses = pd.DataFrame(
    {
        "Gr Liv Area": [1000, 1500, 2500],
        "Bedroom AbvGr": [2, 2, 4],
        "Full Bath": [1, 2, 2]
    },
    index=["House 1", "House 2", "House 3"]
)

reg_models = {
    "Linear regression": linear_reg_model,
    "KNN": knn_reg_model,
    "Voting": voting_reg_model,
    "Stacking": stacking_reg_model
}

house_predictions = example_houses.copy()

for model_name, model in reg_models.items():
    house_predictions[model_name] = model.predict(
        example_houses[reg_features]
    )

house_predictions.round(2)

,Gr Liv Area,Bedroom AbvGr,Full Bath,Linear regression,KNN,Voting,Stacking
House 1,1000,2,1,132024.92,129590.0,130807.46,130552.79
House 2,1500,2,2,217919.89,229259.0,223589.45,226044.10
House 3,2500,4,2,276766.09,279421.3,278093.69,278799.23


In [6]:
from sklearn.model_selection import cross_val_score

reg_cv_results = []

for model_name, model in reg_models.items():
    scores = cross_val_score(
        model,
        X_reg,
        y_reg,
        cv=5,
        scoring="neg_root_mean_squared_error"
    )

    reg_cv_results.append({
        "Model": model_name,
        "Mean CV RMSE": -scores.mean()
    })

reg_cv_scores = (
    pd.DataFrame(reg_cv_results)
    .set_index("Model")
    .sort_values("Mean CV RMSE")
)

reg_cv_scores.round(2)

,Mean CV RMSE
Model,
Stacking,48670.67
Voting,48828.35
KNN,49186.06
Linear regression,51666.76


**Answer:** I used the Ames housing dataset to predict sale price using above-ground living area, number of bedrooms, and number of full bathrooms.

The four models produced different predictions for the example houses. For House 2, linear regression predicted $217,919.89, KNN predicted $229,259.00, voting predicted $223,589.45, and stacking predicted $226,044.10. The voting prediction was the average of the linear regression and KNN predictions.

Using five-fold cross-validation, the mean RMSE was $51,666.76 for linear regression, $49,186.06 for KNN, $48,828.35 for voting, and $48,670.67 for stacking. Stacking performed best under these settings, although its improvement over voting was small. Both ensemble models had lower RMSE than the individual models.

## Voting in Classification

Suppose you have three models for predicting a categorical target:

- k-nearest neighbors
- logistic regression
- decision tree

How might you apply voting to create an ensemble model? Discuss conceptually with your partner. Create a concrete example to illustrate your method. Hint: There are two basic approaches to voting in classification; can you think of both?

Note: You can assume a binary target, but the general idea of voting applies to multiclass problems. However, if any of the candidate *models* are not naturally multiclass (like logistic regression) then you have to consider things like OvR and OvO as we have discussed.

**Answer:** We can combine the KNN, logistic regression, and decision tree models using either hard voting or soft voting.

With hard voting, each model votes for a class, and the class with the most votes is selected. Suppose the three models predict probabilities of 0.60, 0.55, and 0.10 for class 1. Using a 0.5 cutoff, their predicted classes are 1, 1, and 0, so hard voting predicts class 1.

With soft voting, we average the predicted probabilities for each class and select the class with the highest average probability. In the same example, the average probability of class 1 is (0.60 + 0.55 + 0.10) / 3 = 0.417. Therefore, soft voting predicts class 0. This shows that hard and soft voting can produce different predictions.

## Choose your own Classification

**Working together with your partner**, choose an interesting data set to fit *classification* models for, that is, a data set with a *categorical* target variable and several feature variables. Could be a data set that: we have used in class, we have used on an assignment, a data set from your project, you used in another course, or you just find interesting! You'll work together with your partner to develop *classification* models to predict the *categorical* target (so you need to decide on the data set and target variable together with your partner).

After you have decided on the data set and target variable:

- One partner should fit a k-nearest neighbors classification model
- One partner should fit another classification model like decision tree or logistic regression

If you're using an example from earlier in the class, you can use a model you have already come up with or you can try something new. Note: you do NOT have to agree on the features included in your models.

After each partner has their model, **work together to combine your code and create an ensemble model that uses *voting***.

For each of the three models (knn classification, other classification, voting ensemble)

1. Fit the model using the training data
1. Use the fitted model to predict the target for a few observations of your choice, and compare the predictions
1. Estimate the test macro precision, recall, and F1 score using cross-validation, and compare the results

Which model seems to work best?

**Hint:** [`VotingClassifier`](https://scikit-learn.org/1.5/modules/generated/sklearn.ensemble.VotingClassifier.html) works almost identically to `VotingRegressor`.


In [7]:
df_heart = pd.read_csv(
    "https://www.dropbox.com/s/aohbr6yb9ifmc8w/heart_attack.csv?dl=1"
)

class_features = ["age", "trtbps", "chol", "thalach"]

X_class = df_heart[class_features]
y_class = df_heart["output"]

df_heart[class_features + ["output"]].head()

,age,trtbps,chol,thalach,output
0,63,145,233,150,1
1,37,130,250,187,1
2,56,120,236,178,1
3,57,120,354,163,1
4,57,140,192,148,1


In [8]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

knn_class_model = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=10)
)

logistic_class_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

knn_class_model.fit(X_class, y_class)
logistic_class_model.fit(X_class, y_class)

knn_class_model, logistic_class_model

(Pipeline(steps=[('standardscaler', StandardScaler()),
                 ('kneighborsclassifier', KNeighborsClassifier(n_neighbors=10))]),
 Pipeline(steps=[('standardscaler', StandardScaler()),
                 ('logisticregression', LogisticRegression(max_iter=1000))]))

In [9]:
from sklearn.ensemble import VotingClassifier

voting_class_model = VotingClassifier(
    estimators=[
        ("knn", knn_class_model),
        ("logistic", logistic_class_model)
    ],
    voting="soft"
)

voting_class_model.fit(X_class, y_class)

voting_class_model

,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('knn', ...), ('logistic', ...)]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'soft'
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionadded:: 0.18",None
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'


In [10]:
example_patients = df_heart.iloc[[0, 60, 120, 180, 240]].copy()

class_models = {
    "KNN": knn_class_model,
    "Logistic regression": logistic_class_model,
    "Voting": voting_class_model
}

class_predictions = (
    example_patients[class_features + ["output"]]
    .rename(columns={"output": "Actual class"})
    .copy()
)

for model_name, model in class_models.items():
    class_predictions[model_name] = model.predict(
        example_patients[class_features]
    )

class_predictions

,age,trtbps,chol,thalach,Actual class,KNN,Logistic regression,Voting
0,63,145,233,150,1,0,0,0
60,44,120,220,170,1,1,1,1
120,64,128,263,105,1,0,0,0
180,60,125,258,141,0,0,0,0
240,54,122,286,116,0,0,0,0


In [11]:
from sklearn.model_selection import StratifiedKFold, cross_validate

class_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

class_scoring = {
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro"
}

class_cv_results = []

for model_name, model in class_models.items():
    scores = cross_validate(
        model,
        X_class,
        y_class,
        cv=class_cv,
        scoring=class_scoring
    )

    class_cv_results.append({
        "Model": model_name,
        "Macro precision": scores["test_precision"].mean(),
        "Macro recall": scores["test_recall"].mean(),
        "Macro F1": scores["test_f1"].mean()
    })

class_cv_scores = (
    pd.DataFrame(class_cv_results)
    .set_index("Model")
    .sort_values("Macro F1", ascending=False)
)

class_cv_scores.round(3)

,Macro precision,Macro recall,Macro F1
Model,,,
Logistic regression,0.672,0.667,0.666
KNN,0.661,0.659,0.655
Voting,0.652,0.649,0.648


**Answer:** I used the heart attack dataset to predict the binary target `output` using age, resting blood pressure, cholesterol, and maximum heart rate. I compared KNN with 10 neighbors, logistic regression, and a soft voting ensemble that averaged their predicted probabilities.

All three models made the same predictions for the five selected training observations. They correctly classified observations 60, 180, and 240, but misclassified observations 0 and 120.

Using five-fold stratified cross-validation, logistic regression had macro precision of 0.672, recall of 0.667, and F1 of 0.666. KNN had macro precision of 0.661, recall of 0.659, and F1 of 0.655. Voting had macro precision of 0.652, recall of 0.649, and F1 of 0.648.

Logistic regression performed best under these settings. Combining the models with soft voting did not improve the cross-validation results.